### keras_tuner is one of the most famous lib.It is very useful now a days.

In [124]:
import pandas as pd
import numpy as np

In [125]:
df = pd.read_csv('diabetes.csv')

In [126]:
df.head(3)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1


In [127]:
df.shape

(768, 9)

In [128]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [129]:
df = df.drop(columns=['BloodPressure','SkinThickness'])

In [130]:
df.head(3)

,Pregnancies,Glucose,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,0,33.6,0.627,50,1
1,1,85,0,26.6,0.351,31,0
2,8,183,0,23.3,0.672,32,1


In [131]:
from sklearn.model_selection import train_test_split

In [132]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

In [133]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=32)

In [134]:
from sklearn.preprocessing import StandardScaler

In [135]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [136]:
pd.DataFrame(X_train_scaled)

,0,1,2,3,4,5
0,-0.845746,-0.461164,-0.695109,-1.015558,-0.844897,-1.037890
1,0.924554,2.390123,-0.695109,0.992016,-0.055284,0.652018
2,-1.140796,-0.837158,0.157428,-0.445666,0.186984,-0.953395
3,0.629504,1.074144,0.602230,1.807998,0.303631,-0.530918
4,-1.140796,0.384822,-0.695109,0.033562,-0.228759,-1.037890
...,...,...,...,...,...,...
609,1.514654,-1.495147,-0.695109,-0.070055,-0.566738,0.398532
610,3.284955,0.510154,0.324229,0.642310,-0.946589,0.821009
611,0.629504,-1.244485,-0.695109,-0.769468,-0.468036,0.652018
612,0.924554,0.134160,1.297234,-0.860133,-0.922661,0.314037


In [137]:
import keras
import tensorflow

In [138]:
from keras.layers import Dense
from keras import Sequential

In [139]:
model = Sequential()

In [140]:
model.add(Dense(32,activation="relu",input_dim=6))
model.add(Dense(1,activation="sigmoid"))

D:\Deep_Learning\.venv_tf1\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [141]:
model.compile(loss="binary_crossentropy",optimizer="sgd",metrics=['accuracy'])

In [142]:
hist = model.fit(X_train_scaled,y_train,batch_size=32,epochs=10,validation_data=(X_test_scaled,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.3622 - loss: 0.8074 - val_accuracy: 0.3961 - val_loss: 0.8459
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4069 - loss: 0.7678 - val_accuracy: 0.4935 - val_loss: 0.7981
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4997 - loss: 0.7366 - val_accuracy: 0.5714 - val_loss: 0.7592
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6308 - loss: 0.6880 - val_accuracy: 0.6169 - val_loss: 0.7267
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6687 - loss: 0.6714 - val_accuracy: 0.6364 - val_loss: 0.6967
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6931 - loss: 0.6370 - val_accuracy: 0.6494 - val_loss: 0.6709
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7158 - loss: 0.6178 - val_accuracy: 0.6558 - val_loss: 0.6496
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7205 - loss: 0.6036 - val_accuracy: 0.6623 - val_loss

#

#### The problem is we decide on our own how many layers and how many neuron on each layer.
#### we can automate each value using the library keras tuner

## 1.selecting best optimizer of any data

In [143]:
import keras_tuner as kt

In [153]:
# It is from keras-tuner library.
# hp is a class object
def build_model(hp):
    model1 = Sequential()

    model1.add(Dense(32,activation="relu",input_dim=6))
    model1.add(Dense(1,activation="sigmoid"))

    # there are different types of optimizers use the syntax like hp.Choice
    opt = hp.Choice("optimizer", values=['adam','adadelta','rmsprop','sgd'])
    model1.compile(loss="binary_crossentropy",optimizer=opt,metrics=['accuracy'])

    return model1

In [154]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=7)

In [157]:
tuner.search(X_train_scaled,y_train,epochs=7,validation_data=(X_test_scaled,y_test))

In [158]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [159]:
# You don't need to train the model again 
# num_model=1 means top 1 model
model_best = tuner.get_best_models(num_models=1)[0]

D:\Deep_Learning\.venv_tf1\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [160]:
model_best.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape               ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 32)                 │             224 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                  │              33 │
└──────────────────────────────────────┴────────────────────────────┴─────────────────┘

 Total params: 257 (1.00 KB)

 Trainable params: 257 (1.00 KB)

 Non-trainable params: 0 (0.00 B)

In [161]:
model_best.fit(X_train_scaled,y_train,epochs=100,validation_data=(X_test_scaled,y_test),batch_size=32,initial_epoch=7)

Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7256 - loss: 0.5266 - val_accuracy: 0.7857 - val_loss: 0.4586
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7662 - loss: 0.4888 - val_accuracy: 0.8052 - val_loss: 0.4510
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7526 - loss: 0.4953 - val_accuracy: 0.7987 - val_loss: 0.4463
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7604 - loss: 0.4855 - val_accuracy: 0.7922 - val_loss: 0.4431
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7629 - loss: 0.4680 - val_accuracy: 0.7987 - val_loss: 0.4413
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7801 - loss: 0.4566 - val_accuracy: 0.7922 - val_loss: 0.4407
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7695 - loss: 0.4617 - val_accuracy: 0.8052 - val_loss: 0.4413
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7759 - loss: 0.4724 - val_accuracy: 0.8

In [165]:
import shutil
shutil.rmtree('untitled_project')  # or the directory you're using

# Tuning on how many neurons on the layer

In [168]:
def build_model(hp):
    model1 = Sequential()
    # from 8 to 128 neurons like limit
    neurons = hp.Int('units',min_value = 8,max_value = 128,step=8)
    model1.add(Dense(units=neurons,activation="relu",input_dim=6))
    model1.add(Dense(1,activation="sigmoid"))

    model1.compile(loss="binary_crossentropy",optimizer="rmsprop",metrics=['accuracy'])

    return model1

In [169]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=7)

D:\Deep_Learning\.venv_tf1\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [170]:
tuner.search(X_train_scaled,y_train,epochs=10,validation_data=(X_test_scaled,y_test))

Trial 7 Complete [00h 00m 03s]
val_accuracy: 0.7922077775001526

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 23s


In [174]:
tuner.get_best_hyperparameters()[0].values

{'units': 64}

In [175]:
model_best_neurons = tuner.get_best_models(num_models=1)[0]

D:\Deep_Learning\.venv_tf1\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
D:\Deep_Learning\.venv_tf1\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## how to select No of layers

In [178]:
import shutil
shutil.rmtree('untitled_project') 

In [179]:
def build_model(hp):

    model = Sequential()
    model.add(Dense(64,activation="relu",input_dim=6))
    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):

        model.add(Dense(64,activation="relu"))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(loss="binary_crossentropy",optimizer='rmsprop',metrics=['accuracy'])

    return model
        

In [180]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3)

In [181]:
tuner.search(X_train_scaled,y_train,epochs=5,validation_data=(X_test_scaled,y_test))

Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.8116883039474487

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 11s


In [184]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 10}

In [185]:
model_best_layers = tuner.get_best_models(1)[0]

D:\Deep_Learning\.venv_tf1\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
D:\Deep_Learning\.venv_tf1\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [187]:
model_best_layers.fit(X_train,y_train,epochs=10,initial_epoch=6,validation_data=(X_test_scaled,y_test))

Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.5053 - loss: 11.6089 - val_accuracy: 0.6429 - val_loss: 0.5870
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6544 - loss: 0.6575 - val_accuracy: 0.6429 - val_loss: 0.5915
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6234 - loss: 0.6832 - val_accuracy: 0.6429 - val_loss: 0.6036
Epoch 10/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6652 - loss: 0.6701 - val_accuracy: 0.6429 - val_loss: 0.6112


In [188]:
# due to overfitting accuracy decrease
## all at once at next notebook